
# Fine Tuning Qwen 2.5 1.5B using QLoRA
---


In [1]:
!pip install unsloth
!pip install --no-deps bitsandbytes accelerate peft trl
!pip install datasets pandas pyarrow -q

# Uploading dataset
Just for google colab

In [2]:
from google.colab import files
uploaded = files.upload()

Saving cybersecurity_train_clean.jsonl to cybersecurity_train_clean.jsonl


# Fixing role name in JSON
Qwen uses "assistant" not "model"



In [7]:
import json

# Read
with open("cybersecurity_train_clean.jsonl", "r") as f:
    data = [json.loads(line) for line in f]

# Swap "model" → "assistant" for Qwen
for ex in data:
    for msg in ex["messages"]:
        if msg["role"] == "model":
            msg["role"] = "assistant"

# Save patched version
with open("cybersecurity_train_clean.jsonl", "w") as f:
    for ex in data:
        f.write(json.dumps(ex) + "\n")

print(f"✅ Done — roles updated for Qwen ({len(data)} samples)")
print(f"   Roles in first entry: {[m['role'] for m in data[0]['messages']]}")

✅ Done — roles updated for Qwen (2394 samples)
   Roles in first entry: ['system', 'user', 'assistant']


In [17]:
display(data[0])

{'messages': [{'role': 'system',
   'content': 'You are a highly specialized AI assistant for advanced cyber-defense whose mission is to deliver accurate, in-depth, actionable guidance on information-security principles—confidentiality, integrity, availability, authenticity, non-repudiation, and privacy—by offering concise executive summaries that drill down into technical detail, industry standards, and threat models while referencing frameworks such as NIST CSF and MITRE ATT&CK; you may share defensive scripts, detection rules, lab-safe PoC payloads, exploit snippets, and hardening checklists clearly marked for educational/testing use only, redacting or stubbing any data that could cause real harm in production. You must never generate or improve ransomware, wipers, botnets, RATs, phishing kits, social-engineering lures, or any instructions that facilitate fraud, data theft, unauthorized intrusion, or the defeat of security controls—in such cases you must briefly refuse with an apolo

# Config

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional

# qlora or lora
ADAPTER_METHOD = "qlora"


@dataclass
class ModelConfig:
    model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"
    max_seq_length: int = 2048
    load_in_4bit: bool = False
    load_in_8bit: bool = False


@dataclass
class LoRAConfig:
    r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    bias: str = "none"
    use_gradient_checkpointing: str = "unsloth"
    target_modules: List[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ])


@dataclass
class DataConfig:
    train_file: str = "cybersecurity_train_clean.jsonl"
    val_split: float = 0.05             # 5% held out for validation
    dataset_text_field: str = "text"
    seed: int = 42


@dataclass
class TrainingConfig:
    output_dir: str = "./outputs"
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4   # effective batch = 8
    num_train_epochs: int = 3
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.1
    lr_scheduler_type: str = "cosine"
    bf16: bool = False
    fp16: bool = True
    logging_steps: int = 10
    eval_steps: int = 100
    save_strategy: str = "epoch"
    save_total_limit: int = 2
    load_best_model_at_end: bool = True
    report_to: str = "none"              


def get_configs():
    model_cfg = ModelConfig()
    lora_cfg = LoRAConfig()
    data_cfg = DataConfig()
    train_cfg = TrainingConfig()

    if ADAPTER_METHOD == "qlora":
        model_cfg.load_in_4bit = True
        lora_cfg.use_dora = False

    elif ADAPTER_METHOD == "lora":
        model_cfg.load_in_4bit = False
        lora_cfg.use_dora = False

    else:
        raise ValueError(f"Unknown ADAPTER_METHOD: '{ADAPTER_METHOD}'. "
                         f"Choose from: qlora, lora, dora, qdora, loftq, adalora")

    return model_cfg, lora_cfg, data_cfg, train_cfg

# Utils

In [5]:
import random
import numpy as np
import torch
import logging
import os

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_logger(name: str) -> logging.Logger:
    logging.basicConfig(
        format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
        level=logging.INFO,
    )
    return logging.getLogger(name)


def check_gpu():
    logger = get_logger("utils")
    if torch.cuda.is_available():
        gpu = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        logger.info(f"GPU: {gpu} | VRAM: {vram:.1f} GB")
    else:
        logger.warning("No GPU found — training will be very slow on CPU")


def make_output_dir(path: str):
    os.makedirs(path, exist_ok=True)

# Model

In [6]:
from unsloth import FastLanguageModel
from peft import AdaLoraConfig, get_peft_model

logger = get_logger("model")


def load_model(model_cfg: ModelConfig, lora_cfg: LoRAConfig):
    logger.info(f"Loading model: {model_cfg.model_name}")
    logger.info(f"Adapter method: {ADAPTER_METHOD.upper()}")

    # ── Load base model ──
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_cfg.model_name,
        max_seq_length=model_cfg.max_seq_length,
        dtype=None,                          # auto-detect bf16/fp16
        load_in_4bit=model_cfg.load_in_4bit,
        load_in_8bit=model_cfg.load_in_8bit,
    )

    if ADAPTER_METHOD in ("qlora", "lora"):
        logger.info(f"Attaching {'DoRA' if lora_cfg.use_dora else 'LoRA'} adapters "
                    f"(r={lora_cfg.r}, use_dora={lora_cfg.use_dora})")
        model = FastLanguageModel.get_peft_model(
            model,
            r=lora_cfg.r,
            lora_alpha=lora_cfg.lora_alpha,
            lora_dropout=lora_cfg.lora_dropout,
            bias=lora_cfg.bias,
            target_modules=lora_cfg.target_modules,
            use_gradient_checkpointing=lora_cfg.use_gradient_checkpointing,
        )

    else:
        raise ValueError(f"Unknown ADAPTER_METHOD: '{ADAPTER_METHOD}'. "
                         f"Choose from: qlora, lora, loftq, adalora")

    # Print trainable parameter summary
    model.print_trainable_parameters()
    return model, tokenizer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# Dataset

In [8]:
from datasets import load_dataset

logger = get_logger("dataset")


def load_and_split(data_cfg: DataConfig):
    logger.info(f"Loading dataset: {data_cfg.train_file}")

    raw = load_dataset(
        "json",
        data_files=data_cfg.train_file,
        split="train"
    )

    # Train / validation split
    split = raw.train_test_split(
        test_size=data_cfg.val_split,
        seed=data_cfg.seed
    )

    logger.info(f"Train: {len(split['train'])} | Val: {len(split['test'])} examples")
    return split["train"], split["test"]


def format_dataset(train_ds, val_ds, tokenizer):

    def apply_template(example):
        text = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
        return {"text": text}

    logger.info("Applying chat template...")
    train_ds = train_ds.map(apply_template, desc="Formatting train")
    val_ds   = val_ds.map(apply_template,   desc="Formatting val")

    return train_ds, val_ds

# Train

In [9]:
from trl import SFTTrainer, SFTConfig

logger = get_logger("trainer")


def build_trainer(model, tokenizer, train_ds, val_ds,
                  train_cfg, data_cfg):

    args = SFTConfig(
        output_dir=train_cfg.output_dir,
        per_device_train_batch_size=train_cfg.per_device_train_batch_size,
        gradient_accumulation_steps=train_cfg.gradient_accumulation_steps,
        num_train_epochs=train_cfg.num_train_epochs,
        learning_rate=train_cfg.learning_rate,
        warmup_steps=10,
        lr_scheduler_type=train_cfg.lr_scheduler_type,
        bf16=train_cfg.bf16,
        fp16=train_cfg.fp16,
        logging_steps=train_cfg.logging_steps,
        eval_steps=train_cfg.eval_steps,
        eval_strategy="steps",
        save_strategy="steps",
        save_steps=train_cfg.eval_steps,
        save_total_limit=train_cfg.save_total_limit,
        load_best_model_at_end=True,
        report_to=train_cfg.report_to,
        dataset_text_field=data_cfg.dataset_text_field,
        max_seq_length=2048,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        args=args,
    )

    return trainer


def run_training(trainer: SFTTrainer, output_dir: str):
    import os

    resume_from = None
    if os.path.isdir(output_dir):
        checkpoints = [
            os.path.join(output_dir, d)
            for d in os.listdir(output_dir)
            if d.startswith("checkpoint-")
            and os.path.exists(os.path.join(output_dir, d, "trainer_state.json"))
        ]
        if checkpoints:
            resume_from = max(checkpoints, key=lambda x: int(x.split("-")[-1]))
            logger.info(f"Resuming from: {resume_from}")
        else:
            logger.info("No valid checkpoint found — starting from scratch")

    logger.info("Starting training...")
    result = trainer.train(resume_from_checkpoint=resume_from)

    logger.info(f"Training complete.")
    logger.info(f"  Loss:    {result.training_loss:.4f}")
    logger.info(f"  Runtime: {result.metrics['train_runtime']:.0f}s")

    save_path = f"{output_dir}/final_adapter"
    trainer.model.save_pretrained(save_path)
    trainer.processing_class.save_pretrained(save_path)
    logger.info(f"Adapter saved → {save_path}")

# Main

In [19]:
import shutil, os

# ── Clean up any broken checkpoints first ──
if os.path.isdir("./outputs"):
    broken = [
        os.path.join("./outputs", d)
        for d in os.listdir("./outputs")
        if d.startswith("checkpoint-")
        and not os.path.exists(os.path.join("./outputs", d, "trainer_state.json"))
    ]
    for b in broken:
        shutil.rmtree(b)
        print(f"🗑️  Deleted broken checkpoint: {b}")
    if not broken:
        print("✅ No broken checkpoints found")

# ── Load configs ──
model_cfg, lora_cfg, data_cfg, train_cfg = get_configs()

# ── Colab T4 overrides ──
train_cfg.bf16 = False
train_cfg.fp16 = True
train_cfg.output_dir = "./outputs"

# ── Run pipeline ──
check_gpu()
set_seed(data_cfg.seed)

model, tokenizer = load_model(model_cfg, lora_cfg)
train_ds, val_ds = load_and_split(data_cfg)
train_ds, val_ds = format_dataset(train_ds, val_ds, tokenizer)
trainer = build_trainer(model, tokenizer, train_ds, val_ds, train_cfg, data_cfg)
run_training(trainer, train_cfg.output_dir)

==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

...

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,274 | Num Epochs = 3 | Total steps = 855
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
100,1.208145,1.210602
200,1.230305,1.180064
300,1.112880,1.167156
400,1.112655,1.160678
500,1.127340,1.150750
600,1.016327,1.155536
700,1.002643,1.157260
800,1.038302,1.155985
855,1.006435,1.156198


Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-700/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-800/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-855/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/final_adapter/tokenizer_config.json.


# Saving into google drive

In [20]:
from google.colab import drive
import shutil, os

# Mount your Google Drive
drive.mount("/content/drive")

# Create a folder for this specific run
save_dir = "/content/drive/MyDrive/cybersecurity-finetune/Qwen-2.5-1.5B-qlora"
os.makedirs(save_dir, exist_ok=True)

# Copy adapter
shutil.copytree(
    "./outputs/final_adapter",
    f"{save_dir}/final_adapter",
    dirs_exist_ok=True
)

# Copy dataset too while we're at it
shutil.copy(
    "cybersecurity_train_clean.jsonl",
    f"{save_dir}/cybersecurity_train_clean.jsonl"
)

print(f"✅ Saved to Google Drive → {save_dir}")
print(f"   Files: {os.listdir(f'{save_dir}/final_adapter')}")

Mounted at /content/drive
✅ Saved to Google Drive → /content/drive/MyDrive/cybersecurity-finetune/Qwen-2.5-1.5B-qlora
   Files: ['chat_template.jinja', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer_config.json', 'README.md', 'tokenizer.json']


# Uploading from google drive

In [ ]:
from google.colab import drive
from unsloth import FastLanguageModel
import shutil, os

drive.mount("/content/drive")

# Copy back from Drive to Colab
src = "/content/drive/MyDrive/cybersecurity-finetune/Qwen-2.5-1.5B-qlora/final_adapter"
dst = "./outputs/final_adapter"

shutil.copytree(src, dst, dirs_exist_ok=True)
print("✅ Adapter copied from Drive")

# Load it
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=dst,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print("✅ Model ready — no retraining needed!")

# Evaluation

In [ ]:
import json
import torch
import numpy as np
from unsloth import FastLanguageModel
from datasets import load_dataset

# ── Load fine-tuned model ──
model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="/content/drive/MyDrive/cybersecurity-finetune/gemma3-1b-lora/final_adapter",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
model.eval()

# ── Load validation split ──
raw = load_dataset("json", data_files="cybersecurity_train_clean.jsonl", split="train")
split = raw.train_test_split(test_size=0.05, seed=42)
val_ds = split["test"]

print(f"Evaluating on {len(val_ds)} validation samples...")

# ── Compute Perplexity ──
def compute_perplexity(model, tokenizer, dataset, max_length=512, n_samples=100):
    losses = []
    samples = dataset.select(range(min(n_samples, len(dataset))))

    for ex in samples:
        # Format same way as training
        text = tokenizer.apply_chat_template(
            ex["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length
        ).to("cuda")

        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            losses.append(outputs.loss.item())

    avg_loss  = np.mean(losses)
    perplexity = np.exp(avg_loss)
    return avg_loss, perplexity

# ── Run on both models ──
print("\n📊 Computing perplexity on validation set...")

# Fine-tuned
ft_loss, ft_ppl = compute_perplexity(model, tokenizer, val_ds)

# Base model
print("Loading base model for comparison...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="google/gemma-3-1b-it",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
base_model.eval()
base_loss, base_ppl = compute_perplexity(base_model, base_tokenizer, val_ds)

# ── Report ──
improvement_ppl  = ((base_ppl  - ft_ppl)  / base_ppl)  * 100
improvement_loss = ((base_loss - ft_loss) / base_loss) * 100

print(f"\n{'='*50}")
print(f"  EVALUATION RESULTS")
print(f"{'='*50}")
print(f"  {'Metric':<20} {'Base':>10} {'Fine-tuned':>12} {'Δ':>8}")
print(f"  {'-'*48}")
print(f"  {'Cross-entropy Loss':<20} {base_loss:>10.4f} {ft_loss:>12.4f} {improvement_loss:>+7.1f}%")
print(f"  {'Perplexity':<20} {base_ppl:>10.4f} {ft_ppl:>12.4f} {improvement_ppl:>+7.1f}%")
print(f"{'='*50}")

if ft_ppl < base_ppl:
    print(f"\n  ✅ Fine-tuning improved perplexity by {improvement_ppl:.1f}%")
else:
    print(f"\n  ⚠️  Base model has lower perplexity — consider more epochs")